# TabTune · Home Credit · Loan Amount Recommendation
### Tabular foundation models, end to end: data preparation → AutoML → testing → risk → production

**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

| | |
|---|---|
| **Task** | Regression: credit amount (`AMT_CREDIT`) |
| **Data** | Home Credit applications, about 270 features (application, bureau, previous loans, engineered ratios) |
| **Decision supported** | Recommended amount with a safe range |
| **Engine** | TabTune open-source library, running on this notebook's GPU |
| **Production route** | The Lexsi platform (reference cells show the equivalent calls) |

## Notebook map

| # | Section | What you get |
|---|---|---|
| 1 | Onboarding | Install, model options in TabTune and on the Lexsi platform |
| 2 | Product options | Problem statement, training and test data, development vs production choices |
| 3 | Data preparation | Data profile dashboards, a **data preparation agent** with approval gates, feature engineering, conclusions |
| 4 | Model training | AutoML race (foundation models vs tuned classic models), hyperparameter search, selection rule, global explainability |
| 5 | Model testing | Evaluation dashboards on held-out data, interactive threshold, per-decision explanations |
| 6 | Batch inference | Scoring a file end to end, throughput, saved model artefact |
| 7 | Model risk | Uncertainty with guaranteed coverage, fairness, regulatory evidence map |
| 8 | Artefacts before production | Model card written by an open-source LLM with trace and grounding check, production API contract |
| 9 | Model in production | Drift monitoring, **retraining agent** with human approval, regenerated model card |
| 10 | Conclusions | Plain-English summary and the audit trail |

**Run:** *Runtime → Change runtime type → H100 GPU, High-RAM*, then *Run all*. Set `AUTO_APPROVE = False` in the imports cell to be asked at each approval gate.

# 1 · Onboarding

In [ ]:
# ⚙️ Install TabTune (open source) — about 3 minutes on Colab
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError

def _v(pkg):
    try:
        return version(pkg)
    except PackageNotFoundError:
        return None

def pip_install(*packages, required=True):
    """pip install that shows pip's own error message instead of a bare CalledProcessError."""
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stdout + r.stderr)[-4000:])
        if required:
            raise RuntimeError(f"pip install failed for {packages}; pip's message is printed above")
        print(f"(optional) {packages} could not be installed; the notebook continues without it")

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    before = {p: _v(p) for p in ("numpy", "pandas", "scikit-learn")}
    # No [colab] extra: it pins pandas==2.2.2, which has no wheel for Colab's Python 3.13 runtime.
    pip_install("tabtune @ git+https://github.com/Lexsi-Labs/TabTune.git", "plotly", "ipywidgets", "accelerate")
    pip_install("kernels", "triton>=3.4", required=False)    # MXFP4 kernels so gpt-oss-120b stays 4-bit on the H100
    # The Lexsi SDK is NOT installed here: this notebook only shows platform calls as reference text, and the SDK's
    # OpenTelemetry dependencies change Colab's protobuf version, which forces a runtime restart.
    changed = [p for p, v in before.items() if v != _v(p)]
    if changed:
        print(f"⚠️ pip changed {', '.join(changed)} — use Runtime → Restart session, then run all cells again (the second install is instant).")

print(f"python     {sys.version.split()[0]}")
for pkg in ("tabtune", "torch", "numpy", "pandas"):
    print(f"{pkg:10} {_v(pkg) or 'MISSING'}")

In [ ]:
# 🔧 Imports, theme and the run-wide fact sheet
import json, logging, re, time, warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown, HTML
import ipywidgets as W

warnings.filterwarnings("ignore")
from tabtune.logger import setup_logger
setup_logger(level=logging.WARNING)
pd.set_option("display.float_format", lambda v: f"{v:,.4f}")
pd.set_option("display.max_columns", 60)

try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "none"
except ImportError:
    DEVICE, GPU = "cpu", "none"

FAST = os.environ.get("TABTUNE_DEMO_FAST") == "1"      # tiny settings for a smoke test without a GPU
AUTO_APPROVE = True                                    # False = the agents stop and ask you to type yes / no
BLUE, ORANGE, TEAL, GREY, DARK, RED, GOLD = "#2F5BEA", "#B4541A", "#1B998B", "#9AA0A8", "#16181D", "#D7263D", "#E8C547"
PALETTE = [BLUE, ORANGE, TEAL, RED, GOLD, GREY, DARK, "#7A5CFA"]
FACTS = {}                                             # every number that reaches the model card is recorded here
AUDIT = []                                             # every agent step and human decision


def record(section, **values):
    """Store results for the model card and the conclusions."""
    FACTS.setdefault(section, {}).update({k: (round(float(v), 4) if isinstance(v, (int, float, np.floating, np.integer)) and not isinstance(v, bool) else v)
                                          for k, v in values.items()})


def audit(actor, step, finding, decision=""):
    AUDIT.append({"at": pd.Timestamp.now().strftime("%H:%M:%S"), "actor": actor, "step": step, "finding": finding, "decision": decision})
    print(f"[{actor} · {step}] {finding}" + (f"  →  {decision}" if decision else ""))


def human_gate(question, recommended=True):
    """Human-in-the-loop checkpoint. With AUTO_APPROVE the recommendation is taken and logged; otherwise you are asked."""
    if AUTO_APPROVE:
        answer = recommended
        who = "auto-approved (AUTO_APPROVE=True)"
    else:
        reply = input(f"🧑 {question}  [recommended: {'yes' if recommended else 'no'}]  yes/no: ").strip().lower()
        answer = reply.startswith("y") if reply else recommended
        who = "human reviewer"
    audit("human", "approval gate", question, f"{'APPROVED' if answer else 'REJECTED'} · {who}")
    return answer


print(f"device: {DEVICE} ({GPU}) · fast mode: {FAST} · auto-approve: {AUTO_APPROVE}")

In [ ]:
# 📚 Model options in TabTune today (read live from the library's registry)
from tabtune import TabularPipeline
from tabtune.registry import list_models
import logging
for _n in list(logging.root.manager.loggerDict):          # keep the library's progress log out of the demo output
    if "tabtune" in _n.lower():
        logging.getLogger(_n).setLevel(logging.WARNING)

TASK = "regression"
def _lic(spec):
    lic = getattr(spec, "license", None)
    return getattr(lic, "name", "?"), {True: "yes", False: "no", None: "unverified"}[getattr(lic, "commercial_use_ok", None)]

TABTUNE_MODELS = {}
for spec in list_models():
    env = getattr(spec, "envelope", None)
    TABTUNE_MODELS[spec.name] = {
        "family": getattr(spec, "family", ""),
        "classification": bool(spec.classification_strategies), "regression": bool(spec.regression_strategies),
        "strategies": ", ".join(sorted(getattr(spec, f"{TASK}_strategies"))),
        "max_rows": getattr(env, "max_rows", None), "max_features": getattr(env, "max_features", None),
        "max_classes": getattr(env, "max_classes", None),
        "licence": _lic(spec)[0], "commercial_use": _lic(spec)[1], "summary": (spec.summary or "")[:70]}
catalogue = pd.DataFrame(TABTUNE_MODELS).T.rename_axis("model")
usable = catalogue[catalogue[TASK]]
print(f"{len(catalogue)} tabular foundation models registered · {len(usable)} support {TASK} · "
      f"{int((usable.commercial_use == 'yes').sum())} of those are cleared for commercial use")
display(usable.drop(columns=["classification", "regression"]).style.map(
    lambda v: f"color:{TEAL};font-weight:600" if v == "yes" else (f"color:{RED}" if v == "no" else ""), subset=["commercial_use"]))

### The same options on the Lexsi platform
Development happens here with the open-source library. In production the same models run on the Lexsi platform, which adds projects, managed compute, a model registry, case-level audit and policies.

In [ ]:
# 🗂️ The same workflow on the Lexsi platform (reference: nothing in this cell calls the platform)
LEXSI_OPTIONS = {
    "structure": "Organization → Workspace → Project (one project per use case; project type is classification or regression)",
    "data": "project.upload_data(file, tag): tags such as Training / Test / Batch1 select data for training, testing and inference",
    "training_strategies": ["simple", "ensemble", "distillation", "leaderboard"],
    "tuning_strategies": ["inference (zero-shot)", "finetune", "peft (LoRA)"],
    "compute": {"CPU": ["xsmall", "small", "medium", "large", "xlarge", "2xlarge", "3xlarge"],
                "GPU": ["T4.xsmall", "T4.small", "A10G.xmedium"]},
    "foundation_models": ["TabPFN", "TabPFNv26", "TabPFNv3", "TabICL", "TabICLv2", "OrionMSP", "OrionMSPv1.5", "OrionBix", "OrionMoE",
                          "Mitra", "ContextTab", "TabDPT", "Limix", "TabFM", "EXAONETabular", "ILTM", "XRFM"],
    "classic_models": ["XGBoost", "LGBoost", "CatBoost", "RandomForest", "SGD", "LogisticRegression", "LinearRegression", "GaussianNaiveBayes"],
    "explainability": ["shap", "lime", "support_set (training-row attribution, in-context models)"],
    "inference": "project.model_inference(tag=..., model_name=..., compute_type=...) for batches; project.case_predict(...) for one case",
    "governance": "model registry per project, activate_model for promotion, case audit trail, policies and observations",
}
display(pd.DataFrame({"On the Lexsi platform": {k: (", ".join(v) if isinstance(v, list) else json.dumps(v) if isinstance(v, dict) else v)
                                               for k, v in LEXSI_OPTIONS.items()}}).style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))
display(Markdown("""
**Platform equivalent of this notebook's training step** (reference only; run it in a separate environment after `pip install lexsi-sdk`):
```python
from lexsi_sdk import lexsi
lexsi.login()
project = lexsi.organization("…").workspace("…").project("…")
project.upload_data("train.csv", "Training", config={"unique_identifier": "id", "true_label": "TARGET"}, compute_type="small")
project.train_model(model="OrionMSP", training_strategy="simple", tunning_strategy="inference", compute_type="T4.small",
                    data_config={"tags": ["Training"], "test_tags": ["Test"], "xai_method": ["shap", "lime"]})
```"""))

# 2 · Product options

## Problem statement
**How much credit should this applicant be offered?** A retail lender recommends a loan amount from the applicant's profile, bureau history and previous applications, to support limit setting and affordability checks.

## Training and test data
The notebook draws four disjoint sets from the source data, so nothing used to choose a model is used to grade it:

| Set | Rows (A100 default) | Used for |
|---|---|---|
| **Context (training)** | 5,000 | the rows a foundation model reads as context; training data for classic models |
| **Validation** | 2,000 | model selection, threshold choice, conformal calibration |
| **Test** | 3,000 | final, untouched performance report |
| **Production stream** | 6 × 1,000 | six later "months" for drift monitoring and retraining |

## Development vs production

| Choice | In development (this notebook) | In production (Lexsi platform) |
|---|---|---|
| Engine | TabTune open source on the notebook GPU | Lexsi-managed TabTune build with proprietary engines |
| Models | every model in the registry above | 17 foundation models plus classic AutoML |
| Adaptation | zero-shot, fine-tuning, LoRA | same, as managed training jobs |
| Scoring | batch scoring in the notebook | batch by tag, single case through the API |
| Governance | audit list in this notebook | registry, approvals, case audit trail, policies |

In [ ]:
# 🔧 Use-case settings
TARGET_NAME = "credit amount"
N_CONTEXT, N_VALID, N_TEST, N_MONTH, MONTHS = (400, 200, 300, 150, 4) if FAST else (5000, 2000, 3000, 1000, 6)
ROSTER = ["TabICLv2", "TabPFN", "Mitra", "TabDPT"]                                   # foundation models raced zero-shot
if FAST:
    ROSTER = ROSTER[:2]
REQUIRE_COMMERCIAL = False                            # True = only models cleared for commercial use may win
RUN_PEFT = RUN_ENSEMBLE = (not FAST) or os.environ.get("TABTUNE_DEMO_PATHS") == "1"
TOP_K_FEATURES = 20 if FAST else 60                   # the preparation agent proposes keeping the strongest features
LORA_EPOCHS = 1 if FAST else 3
SEED = 7

# 3 · Data preparation

## 3.1 Load

In [ ]:
# 📥 Home Credit (processed parquet files on Google Drive)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except ImportError:
    pass
DATA_DIR = os.environ.get("HC_DATA_DIR", "/content/drive/MyDrive/datasets/home-credit-default-risk/processed")
data = pd.concat([pd.read_parquet(os.path.join(DATA_DIR, f)) for f in ("X_train.parquet", "X_val.parquet")], ignore_index=True)
data.columns = [re.sub(r"[^0-9A-Za-z_]+", "_", str(c)).strip("_") for c in data.columns]
data = data.loc[:, ~data.columns.duplicated()]
UID, TARGET_COL = "SK_ID_CURR", "AMT_CREDIT"
data[TARGET_COL] = pd.to_numeric(data[TARGET_COL], errors="coerce")
data = data[data[TARGET_COL].notna()].reset_index(drop=True)
# Fields that restate the answer or are only known after the loan exists
KNOWN_EXCLUDE = {c: "leaks the target (derived from or equal to the credit amount)" for c in data.columns
                 if c in ("AMT_GOODS_PRICE", "AMT_ANNUITY") or (c.startswith("FE_") and re.search("CREDIT|ANNUITY|GOODS", c))}
if "TARGET" in data:
    KNOWN_EXCLUDE["TARGET"] = "default outcome, observed only after the loan"
CLASS_NAMES = None
GROUP_PREFIX, GROUP_LABEL = "CODE_GENDER", "gender"
print(f"{len(data):,} applications · {data.shape[1] - 2} candidate features · median credit {data[TARGET_COL].median():,.0f}")
display(data.sample(5, random_state=1).iloc[:, :12])

In [ ]:
# ✂️ Four disjoint sets: context, validation, test and a production stream
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(SEED)
need = N_CONTEXT + N_VALID + N_TEST + N_MONTH * MONTHS
work = data.sample(min(len(data), max(need * 3, need)), random_state=SEED).reset_index(drop=True)
strat = work[TARGET_COL] if TASK == "classification" else pd.qcut(work[TARGET_COL].rank(method="first"), 10, labels=False)
dev, stream = train_test_split(work, train_size=N_CONTEXT + N_VALID + N_TEST, stratify=strat, random_state=SEED)
s2 = strat.loc[dev.index]
ctx, rest = train_test_split(dev, train_size=N_CONTEXT, stratify=s2, random_state=SEED)
val, test = train_test_split(rest, train_size=N_VALID, stratify=s2.loc[rest.index], random_state=SEED)
ctx, val, test, stream = (d.reset_index(drop=True) for d in (ctx, val, test, stream))
ALL_FEATURES = [c for c in data.columns if c not in (UID, TARGET_COL)]
sizes = pd.Series({"context (training)": len(ctx), "validation": len(val), "test": len(test), "production stream": len(stream)})
fig = make_subplots(rows=1, cols=2, column_widths=[0.45, 0.55], subplot_titles=("Rows per set", f"Target in each set ({TARGET_NAME})"))
fig.add_trace(go.Bar(x=sizes.index, y=sizes.values, marker_color=[BLUE, TEAL, ORANGE, GREY], text=sizes.values, textposition="outside", showlegend=False), 1, 1)
for name, d, col in (("context", ctx, BLUE), ("validation", val, TEAL), ("test", test, ORANGE)):
    if TASK == "classification":
        fig.add_trace(go.Bar(x=[name], y=[d[TARGET_COL].mean() * 100], marker_color=col, showlegend=False, text=[f"{d[TARGET_COL].mean():.1%}"], textposition="outside"), 1, 2)
    else:
        fig.add_trace(go.Violin(y=d[TARGET_COL], name=name, line_color=col, box_visible=True, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=420, title="Data split · stratified so every set has the same target mix")
fig.show()
record("data", source="Lending Club" if TASK == "classification" else "Home Credit", rows_available=len(data), context_rows=len(ctx),
       validation_rows=len(val), test_rows=len(test), candidate_features=len(ALL_FEATURES), target=TARGET_COL)

## 3.2 Data quality and distributions

In [ ]:
# 🔎 Profile dashboard: types, missing values, cardinality, skew
NUM = [c for c in ALL_FEATURES if pd.api.types.is_numeric_dtype(ctx[c])]
CAT = [c for c in ALL_FEATURES if c not in NUM]
miss = ctx[ALL_FEATURES].isna().mean().mul(100)
uniq = ctx[ALL_FEATURES].nunique()
const = [c for c in ALL_FEATURES if uniq[c] <= 1]
binary = [c for c in NUM if ctx[c].dropna().isin([0, 1]).all() and c not in const]
cont = [c for c in NUM if c not in binary and uniq[c] > 20]
skew = ctx[cont].skew().abs().sort_values(ascending=False) if cont else pd.Series(dtype=float)

kinds = pd.Series({"continuous": len(cont), "binary / one-hot": len(binary), "low-cardinality numeric": len(NUM) - len(cont) - len(binary), "categorical": len(CAT)})
fig = make_subplots(rows=2, cols=3, vertical_spacing=0.2, specs=[[{"type": "domain"}, {}, {}], [{}, {}, {}]],
                    subplot_titles=("Feature types", "Most incomplete features (% missing)", "Distinct values per feature (log)",
                                    "Most skewed features", "Heaviest-tailed feature · raw", "Same feature · signed log"))
fig.add_trace(go.Pie(labels=kinds.index, values=kinds.values, hole=0.55, marker_colors=[BLUE, TEAL, GOLD, ORANGE], showlegend=False, textinfo="label+value"), 1, 1)
tm = miss.sort_values(ascending=False).head(12)[::-1]
fig.add_trace(go.Bar(x=tm.values, y=[c[:26] for c in tm.index], orientation="h", marker_color=[RED if v > 50 else GREY for v in tm.values], showlegend=False), 1, 2)
fig.add_trace(go.Histogram(x=np.log10(uniq.clip(lower=1)), nbinsx=30, marker_color=BLUE, showlegend=False), 1, 3)
if len(skew):
    sk = skew.head(10)[::-1]; f0 = skew.index[0]
    fig.add_trace(go.Bar(x=sk.values, y=[c[:26] for c in sk.index], orientation="h", marker_color=ORANGE, showlegend=False), 2, 1)
    fig.add_trace(go.Histogram(x=ctx[f0], nbinsx=60, marker_color=GREY, showlegend=False), 2, 2)
    fig.add_trace(go.Histogram(x=np.sign(ctx[f0]) * np.log1p(ctx[f0].abs()), nbinsx=60, marker_color=TEAL, showlegend=False), 2, 3)
fig.update_layout(template="plotly_white", height=720, title="Data profile of the context set")
fig.show()
record("data", features_with_missing=int((miss > 0).sum()), constant_features=len(const), categorical_features=len(CAT),
       heavily_skewed_features=int((skew > 2).sum()))

In [ ]:
# 🎯 Signal dashboard: which features move the target, and which repeat each other
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression

def encode(df, cols=None):
    """Numeric view of a frame: categoricals become stable integer codes, missing values stay missing."""
    cols = cols or ALL_FEATURES
    out = df[cols].copy()
    for c in cols:
        if c in CAT_CODES:
            out[c] = out[c].map(CAT_CODES[c]).astype(float)
    return out.replace([np.inf, -np.inf], np.nan)

CAT_CODES = {c: {v: i for i, v in enumerate(sorted(data[c].dropna().astype(str).unique()))} for c in CAT}
for c in CAT:
    for d in (ctx, val, test, stream):
        d[c] = d[c].astype(str).where(d[c].notna(), None)
Xc_num = encode(ctx)
fill = Xc_num.median()
mi_fn = mutual_info_classif if TASK == "classification" else mutual_info_regression
MI = pd.Series(mi_fn(Xc_num.fillna(fill).fillna(0), ctx[TARGET_COL], random_state=0), index=ALL_FEATURES).sort_values(ascending=False)
top = MI.head(16).index.tolist()
corr = Xc_num[top].corr()
tcorr = Xc_num.corrwith(ctx[TARGET_COL].astype(float)).abs()

fig = make_subplots(rows=1, cols=2, column_widths=[0.38, 0.62], subplot_titles=("Mutual information with the target (top 16)", "Correlation among those features"))
fig.add_trace(go.Bar(x=MI.head(16).values[::-1], y=[c[:28] for c in top[::-1]], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
fig.add_trace(go.Heatmap(z=corr.values, x=[c[:16] for c in top], y=[c[:16] for c in top], colorscale="RdBu_r", zmin=-1, zmax=1), 1, 2)
fig.update_layout(template="plotly_white", height=600, title="Signal and redundancy")
fig.show()

fig = make_subplots(rows=1, cols=4, subplot_titles=[c[:26] for c in top[:4]])
for j, c in enumerate(top[:4], 1):
    b = pd.qcut(Xc_num[c].rank(method="first"), 10, labels=False)
    g = pd.DataFrame({"b": b, "x": Xc_num[c], "y": ctx[TARGET_COL]}).groupby("b").agg(x=("x", "median"), y=("y", "mean"))
    fig.add_trace(go.Scatter(x=g.x, y=g.y * (100 if TASK == "classification" else 1), mode="lines+markers", line=dict(color=ORANGE, width=3), showlegend=False), 1, j)
fig.update_yaxes(title_text="default rate (%)" if TASK == "classification" else f"mean {TARGET_COL}", row=1, col=1)
fig.update_layout(template="plotly_white", height=360, title="How the target responds across deciles of the four strongest features")
fig.show()

## 3.3 Data preparation agent

A small rule-based agent inspects the profile above, **proposes** one action per finding, waits at a **human approval gate**, applies what was approved, and logs everything. It never sees the validation or test sets.

In [ ]:
# 🤖 Data preparation agent: inspect → propose → approve → apply → log
class DataPrepAgent:
    def __init__(self, frame, features):
        self.frame, self.features, self.steps, self.dropped = frame, list(features), [("candidate features", len(features))], {}

    def _propose(self, name, cols, reason, recommended=True):
        cols = [c for c in cols if c in self.features]
        if not cols:
            audit("prep-agent", name, "nothing to do")
            return
        audit("prep-agent", name, f"{len(cols)} feature(s): {', '.join(cols[:5])}{'…' if len(cols) > 5 else ''} — {reason}")
        if human_gate(f"{name}: remove {len(cols)} feature(s)?", recommended):
            self.features = [c for c in self.features if c not in cols]
            self.dropped.update({c: name for c in cols})
        self.steps.append((name, len(self.features)))

    def run(self):
        f = self.frame
        self._propose("known exclusions", list(KNOWN_EXCLUDE), "identifiers or fields flagged by the data owner")
        self._propose("constant columns", const, "one value only, no information")
        self._propose("mostly missing", miss[miss > 80].index.tolist(), "more than 80% missing")
        ident = [c for c in self.features if c in CAT and uniq[c] > 0.5 * len(f)]
        self._propose("identifier-like text", ident, "almost one distinct value per row")
        leak = tcorr[(tcorr > 0.95)].index.tolist() if TASK == "regression" else tcorr[tcorr > 0.98].index.tolist()
        self._propose("target leakage", leak, "near-perfect relationship with the target; it would not exist at decision time")
        # redundant twins among the stronger features
        strong = [c for c in MI.index if c in self.features and c in NUM][:150]
        cm = Xc_num[strong].corr().abs()
        twins = []
        for i, a in enumerate(strong):
            if a in twins:
                continue
            twins += [b for b in strong[i + 1:] if b not in twins and cm.loc[a, b] > 0.97]
        self._propose("redundant twins", twins, "correlated above 0.97 with a stronger feature")
        weak = [c for c in MI.index if c in self.features][TOP_K_FEATURES:]
        self._propose(f"keep the {TOP_K_FEATURES} strongest", weak, "little or no signal; fewer features means faster scoring and clearer explanations")
        return self

prep_agent = DataPrepAgent(ctx, ALL_FEATURES).run()
FEATURES = prep_agent.features
steps = pd.DataFrame(prep_agent.steps, columns=["step", "features left"])
why = pd.Series(prep_agent.dropped).value_counts()
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], specs=[[{}, {"type": "domain"}]], subplot_titles=("Features remaining after each agent step", "Why features were removed"))
fig.add_trace(go.Funnel(y=steps.step, x=steps["features left"], marker_color=BLUE, textinfo="value", showlegend=False), 1, 1)
fig.add_trace(go.Pie(labels=why.index, values=why.values, hole=0.5, marker_colors=PALETTE, showlegend=False, textinfo="label+value"), 1, 2)
fig.update_layout(template="plotly_white", height=520, title=f"Data preparation agent · {len(ALL_FEATURES)} → {len(FEATURES)} features")
fig.show()
record("preparation", features_before=len(ALL_FEATURES), features_after_agent=len(FEATURES), features_removed=len(prep_agent.dropped),
       leakage_features_removed=int((pd.Series(prep_agent.dropped) == "target leakage").sum()) + len(KNOWN_EXCLUDE))

## 3.4 Automated feature engineering

In [ ]:
# 🛠️ Generate candidate features, keep the ones a probe model says help
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import cross_val_score

base = [c for c in MI.index if c in FEATURES and c in cont][:8]
recipes = []
for i, a in enumerate(base):
    for b in base[i + 1:]:
        recipes += [(f"fe_prod__{a}__{b}", "prod", a, b), (f"fe_diff__{a}__{b}", "diff", a, b)]
        if (ctx[b] > 0).all():
            recipes.append((f"fe_ratio__{a}__{b}", "ratio", a, b))

def apply_recipes(df, recs):
    out = {}
    for name, op, a, b in recs:
        out[name] = df[a] * df[b] if op == "prod" else df[a] - df[b] if op == "diff" else df[a] / df[b].where(df[b] > 0)
    return pd.DataFrame(out, index=df.index).replace([np.inf, -np.inf], np.nan)

cand = apply_recipes(ctx, recipes)
cand_mi = pd.Series(mi_fn(cand.fillna(cand.median()).fillna(0), ctx[TARGET_COL], random_state=0), index=cand.columns) if len(recipes) else pd.Series(dtype=float)
parent = pd.Series({n: max(MI[a], MI[b]) for n, _, a, b in recipes})
gain = (cand_mi - parent).sort_values(ascending=False)
KEPT = gain.head(8).index.tolist()
Probe = HistGradientBoostingRegressor if TASK == "regression" else HistGradientBoostingClassifier
scoring = "r2" if TASK == "regression" else "roc_auc"
cv = lambda X: cross_val_score(Probe(max_iter=120, random_state=0), X, ctx[TARGET_COL], cv=3, scoring=scoring).mean()
base_X = encode(ctx, FEATURES)
score_raw, score_fe = cv(base_X), (cv(pd.concat([base_X, cand[KEPT]], axis=1)) if KEPT else np.nan)
helps = bool(KEPT) and score_fe > score_raw + 0.0005
audit("prep-agent", "feature engineering", f"probe {scoring} {score_raw:.4f} → {score_fe:.4f} with {len(KEPT)} engineered features",
      "worth adding" if helps else "no gain: keep raw features")
ADD_ENGINEERED = helps and human_gate(f"Add {len(KEPT)} engineered features?", True)
KEPT_RECIPES = [r for r in recipes if r[0] in KEPT] if ADD_ENGINEERED else []
if ADD_ENGINEERED:
    for d in (ctx, val, test, stream):
        fe = apply_recipes(d, KEPT_RECIPES)
        for c in fe.columns:
            d[c] = fe[c]
    FEATURES = FEATURES + KEPT
    MI = pd.concat([MI, cand_mi[KEPT]]).sort_values(ascending=False)

fig = make_subplots(rows=1, cols=3, column_widths=[0.36, 0.4, 0.24],
                    subplot_titles=(f"{len(recipes)} candidates: signal vs best parent", "Best candidates: gain over parents", f"Probe model ({scoring}, 3-fold)"))
if len(recipes):
    fig.add_trace(go.Scatter(x=parent, y=cand_mi, mode="markers", showlegend=False,
                             marker=dict(color=[ORANGE if n in KEPT else GREY for n in cand_mi.index], size=[10 if n in KEPT else 5 for n in cand_mi.index])), 1, 1)
    g = gain[KEPT][::-1]
    fig.add_trace(go.Bar(x=g.values, y=[n.replace("fe_", "").replace("__", " · ")[:40] for n in g.index], orientation="h", marker_color=ORANGE, showlegend=False), 1, 2)
fig.add_trace(go.Bar(x=["raw", "+ engineered"], y=[score_raw, score_fe], marker_color=[GREY, ORANGE], text=[f"{score_raw:.4f}", f"{score_fe:.4f}"], textposition="outside", showlegend=False), 1, 3)
fig.update_yaxes(range=[np.nanmin([score_raw, score_fe]) * 0.97, np.nanmax([score_raw, score_fe]) * 1.02], row=1, col=3)
fig.update_layout(template="plotly_white", height=520, title="Automated feature engineering · " + ("added to the feature set" if ADD_ENGINEERED else "not added: no measurable gain"))
fig.show()
record("features", final_feature_count=len(FEATURES), engineered_features_added=len(KEPT_RECIPES), strongest_features=", ".join([c for c in MI.index if c in FEATURES][:5]))

In [ ]:
# 🗺️ The prepared data in two dimensions, and the model-ready frames
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
Xtr, ytr = ctx[FEATURES], ctx[TARGET_COL]
Xva, yva = val[FEATURES], val[TARGET_COL]
Xte, yte = test[FEATURES], test[TARGET_COL]
E = encode(ctx, FEATURES); FILL = E.median()
Z = PCA(n_components=2, random_state=0).fit_transform(StandardScaler().fit_transform(E.fillna(FILL).fillna(0)))
fig = go.Figure()
if TASK == "classification":
    for cls, col in ((0, BLUE), (1, RED)):
        m = (ytr == cls).to_numpy()
        fig.add_trace(go.Scattergl(x=Z[m, 0], y=Z[m, 1], mode="markers", name=CLASS_NAMES[cls], marker=dict(size=4, color=col, opacity=0.45)))
else:
    fig.add_trace(go.Scattergl(x=Z[:, 0], y=Z[:, 1], mode="markers", marker=dict(size=4, color=np.log1p(ytr), colorscale="Viridis", opacity=0.6, colorbar=dict(title="log target"))))
fig.update_layout(template="plotly_white", height=480, title="Context set after preparation (PCA projection)")
fig.show()
display(Markdown(f"""
### Data preparation: conclusions
- **{len(ALL_FEATURES)} candidate features became {len(FEATURES)}** after the agent's approved steps; {FACTS['preparation']['leakage_features_removed']} were removed as leakage or known exclusions.
- **{FACTS['data']['features_with_missing']} features have missing values** and {FACTS['data']['heavily_skewed_features']} are heavily skewed; the foundation models apply their own robust scaling and missing-value handling, so no manual imputation was needed.
- **Engineered features were {'added' if ADD_ENGINEERED else 'not added'}** (probe {scoring} {score_raw:.4f} → {score_fe:.4f}).
- Every step is in the audit trail at the end of the notebook.
"""))

# 4 · Model training

## 4.1 AutoML race
Every candidate is fitted on the **context** set and scored on the **validation** set. Foundation models run **zero-shot**: they read the context rows and predict, with no gradient training. Classic models are trained, and XGBoost gets a hyperparameter search.

`TabularLeaderboard(X_train, X_test, y_train, y_test).add_all().run()` does the same race in one line; the loop below keeps each fitted pipeline so the later sections can reuse it.

In [ ]:
# 🏁 Race: zero-shot foundation models vs classic models
from sklearn import metrics as skm
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
import xgboost as xgb

PRIMARY = "roc_auc" if TASK == "classification" else "r2"

def score_of(model, X):
    """Probability of the positive class, or the predicted value."""
    if TASK == "classification":
        p = np.asarray(model.predict_proba(X))
        return p[:, 1] if p.ndim == 2 else p
    return np.asarray(model.predict(X), dtype=float)

def metric(y, s):
    return float(skm.roc_auc_score(y, s) if TASK == "classification" else skm.r2_score(y, s))

class Tabular:                                         # classic models see the numeric encoding
    def __init__(self, est): self.est = est
    def fit(self, X, y): self.est.fit(encode(X, list(X.columns)), y); return self
    def predict(self, X): return self.est.predict(encode(X, list(X.columns)))
    def predict_proba(self, X): return self.est.predict_proba(encode(X, list(X.columns)))

def fit_tfm(name, X, y, strategy="inference", tuning_params=None, **kw):
    pipe = TabularPipeline(model_name=name, task_type=TASK, tuning_strategy=strategy,
                           tuning_params={"device": DEVICE, **(tuning_params or {})}, **kw)
    pipe.fit(X, y)
    return pipe

RACE, FITTED, VAL_SCORES = [], {}, {}
def enter(name, family, build, tuning="none"):
    t0 = time.time()
    try:
        model = build()
        fit_s = time.time() - t0
        t1 = time.time(); s = score_of(model, Xva); pred_s = time.time() - t1
        FITTED[name], VAL_SCORES[name] = model, s
        spec = TABTUNE_MODELS.get(name, {})
        RACE.append({"model": name, "family": family, "tuning": tuning, PRIMARY: metric(yva, s), "fit_seconds": fit_s,
                     "ms_per_row": pred_s / len(Xva) * 1000, "commercial_use": spec.get("commercial_use", "yes"), "status": "ok"})
        print(f"✅ {name:22} {PRIMARY} {RACE[-1][PRIMARY]:.4f} · fit {fit_s:.1f}s")
    except Exception as exc:
        RACE.append({"model": name, "family": family, "tuning": tuning, "status": f"failed: {str(exc)[:90]}"})
        print(f"❌ {name:22} {str(exc)[:110]}")

for name in ROSTER:
    enter(name, "Foundation model (zero-shot)", lambda n=name: fit_tfm(n, Xtr, ytr), "none (zero-shot)")
Base = (lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=500))) if TASK == "classification" \
       else (lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()))
enter("Linear baseline", "Classic ML", lambda: Tabular(Base()).fit(Xtr, ytr))
enter("HistGradientBoosting", "Classic ML", lambda: Tabular(Probe(max_iter=300, random_state=0)).fit(Xtr, ytr), "defaults")

## 4.2 Hyperparameter tuning
Classic models need a search; here XGBoost is tuned with a randomised search and 3-fold cross-validation on the context set. Foundation models need no search for zero-shot use; their tuning knob is **adaptation**, tested next with LoRA fine-tuning.

In [ ]:
# 🔬 XGBoost hyperparameter search, then LoRA fine-tuning of the best foundation model
from sklearn.model_selection import RandomizedSearchCV
XGB = xgb.XGBClassifier if TASK == "classification" else xgb.XGBRegressor
space = {"n_estimators": [200, 400, 700], "max_depth": [3, 4, 6, 8], "learning_rate": [0.02, 0.05, 0.1], "subsample": [0.7, 0.85, 1.0],
         "colsample_bytree": [0.5, 0.75, 1.0], "min_child_weight": [1, 5, 10]}
search = RandomizedSearchCV(XGB(tree_method="hist", random_state=0), space, n_iter=4 if FAST else 20, cv=3, scoring=scoring, random_state=0, n_jobs=2)
search.fit(encode(Xtr, FEATURES), ytr)
trials = pd.DataFrame(search.cv_results_["params"]).assign(score=search.cv_results_["mean_test_score"])
enter("XGBoost (tuned)", "Classic ML", lambda: Tabular(XGB(tree_method="hist", random_state=0, **search.best_params_)).fit(Xtr, ytr), f"random search, {len(trials)} trials")
fig = px.parallel_coordinates(trials, color="score", color_continuous_scale="Viridis",
                              title=f"XGBoost search · {len(trials)} trials · each line is one configuration, colour = cross-validated {scoring}")
fig.update_layout(height=440); fig.show()

tfm_ok = [r for r in RACE if r["status"] == "ok" and r["family"].startswith("Foundation")]
if RUN_PEFT and tfm_ok:
    lora_ok = [r for r in tfm_ok if "peft" in TABTUNE_MODELS.get(r["model"], {}).get("strategies", "")]
    if lora_ok:
        best_tfm = max(lora_ok, key=lambda r: r[PRIMARY])["model"]
        enter(f"{best_tfm} + LoRA", "Foundation model (fine-tuned)",
              lambda: fit_tfm(best_tfm, Xtr, ytr, strategy="peft", tuning_params={"epochs": LORA_EPOCHS, "learning_rate": 2e-5}), f"LoRA, {LORA_EPOCHS} epoch(s)")
    else:
        print(f"None of the raced foundation models has a LoRA path for {TASK} in this library version; the zero-shot results stand.")

## 4.3 Model selection
**Rule:** highest validation score among eligible models; models within 0.002 of the best are treated as tied and the faster one wins. Eligibility can require commercially usable weights (`REQUIRE_COMMERCIAL`). The test set is not used here.

In [ ]:
# 🏆 Leaderboard, selection rule, and an optional ensemble
if RUN_ENSEMBLE and len(tfm_ok) >= 2:
    from tabtune.ensemble import TabularEnsemble
    members = [r["model"] for r in sorted(tfm_ok, key=lambda r: -r[PRIMARY])[:3]]
    enter("TabTune ensemble", "Ensemble",
          lambda: TabularEnsemble(models=[{"model_name": m, "tuning_strategy": "inference", "tuning_params": {"device": DEVICE}} for m in members],
                                  ensemble_strategy="greedy_selection", task_type=TASK, verbose=False,
                                  **({"metric": "r2"} if TASK == "regression" else {})).fit(Xtr, ytr), "greedy selection of " + ", ".join(members))

board = pd.DataFrame(RACE)
ok = board[board.status == "ok"].sort_values(PRIMARY, ascending=False).reset_index(drop=True)
eligible = ok[(ok.commercial_use != "no")] if REQUIRE_COMMERCIAL else ok
tied = eligible[eligible[PRIMARY] >= eligible[PRIMARY].max() - 0.002]
CHAMPION = tied.sort_values("ms_per_row").iloc[0]["model"]
champion = FITTED[CHAMPION]
FAM = {"Foundation model (zero-shot)": BLUE, "Foundation model (fine-tuned)": TEAL, "Classic ML": GREY, "Ensemble": ORANGE}
fig = make_subplots(rows=1, cols=2, subplot_titles=(f"Validation {PRIMARY}", "Accuracy vs scoring cost"))
o = ok.sort_values(PRIMARY)
fig.add_trace(go.Bar(x=o[PRIMARY], y=o.model, orientation="h", marker_color=[FAM[f] for f in o.family],
                     marker_line=dict(color=[DARK if m == CHAMPION else "rgba(0,0,0,0)" for m in o.model], width=3),
                     text=o[PRIMARY], texttemplate="%{text:.4f}", showlegend=False), 1, 1)
for fam, g in ok.groupby("family"):
    fig.add_trace(go.Scatter(x=g.ms_per_row, y=g[PRIMARY], mode="markers+text", text=g.model, textposition="top center", name=fam, marker=dict(size=14, color=FAM[fam])), 1, 2)
fig.update_xaxes(range=[ok[PRIMARY].min() - 0.03, ok[PRIMARY].max() + 0.02], row=1, col=1)
fig.update_xaxes(type="log", title_text="milliseconds per scored row", row=1, col=2)
fig.update_layout(template="plotly_white", height=520, title=f"AutoML leaderboard · selected: {CHAMPION}", legend=dict(orientation="h", y=-0.15))
fig.show()
display(board.style.format(precision=4, na_rep="-").hide(axis="index"))
audit("automl", "model selection", f"{len(ok)} models scored on validation; best {ok.iloc[0]['model']} at {ok.iloc[0][PRIMARY]:.4f}", f"SELECTED {CHAMPION}")
record("selection", candidates_raced=len(board), candidates_succeeded=len(ok), selected_model=CHAMPION, selection_metric=PRIMARY,
       selected_validation_score=float(ok.set_index("model").loc[CHAMPION, PRIMARY]), best_classic_model=str(ok[ok.family == "Classic ML"].iloc[0]["model"]) if (ok.family == "Classic ML").any() else "none",
       best_classic_validation_score=float(ok[ok.family == "Classic ML"][PRIMARY].max()) if (ok.family == "Classic ML").any() else float("nan"),
       selection_rule="highest validation score; ties within 0.002 go to the faster model", xgboost_trials=len(trials))
record("training", context_rows=len(Xtr), features=len(FEATURES), adaptation=str(ok.set_index("model").loc[CHAMPION, "tuning"]), device=GPU)

## 4.4 Global explainability
Permutation importance: shuffle one feature at a time on validation rows and measure how much the score falls. It is model-agnostic, so the same method explains a foundation model and a tree. (`pipeline.get_feature_importance()` is TabTune's built-in version.)

In [ ]:
# 🔍 What drives the selected model overall
def permutation_importance_of(model, X, y, feats, repeats=2, n=300):
    idx = X.sample(min(n, len(X)), random_state=0).index
    Xs, ys = X.loc[idx], y.loc[idx]
    base = metric(ys, score_of(model, Xs))
    out = {}
    for f in feats:
        drops = []
        for r in range(repeats):
            Xp = Xs.copy(); Xp[f] = Xs[f].sample(frac=1, random_state=r).to_numpy()
            drops.append(base - metric(ys, score_of(model, Xp)))
        out[f] = float(np.mean(drops))
    return pd.Series(out).sort_values(ascending=False), base

EXPLAIN_FEATURES = [c for c in MI.index if c in FEATURES][:8 if FAST else 20]
gimp, _ = permutation_importance_of(champion, Xva, yva, EXPLAIN_FEATURES)
t = gimp.head(15)[::-1]
fig = make_subplots(rows=1, cols=2, column_widths=[0.5, 0.5], subplot_titles=(f"Drop in {PRIMARY} when a feature is shuffled", "Model importance vs raw signal (rank agreement)"))
fig.add_trace(go.Bar(x=t.values, y=[c[:30] for c in t.index], orientation="h", marker_color=BLUE, showlegend=False), 1, 1)
ranks = pd.DataFrame({"model": gimp.rank(ascending=False), "signal": MI[gimp.index].rank(ascending=False)})
fig.add_trace(go.Scatter(x=ranks.signal, y=ranks.model, mode="markers+text", text=[c[:14] for c in ranks.index], textposition="top center", marker=dict(size=9, color=ORANGE), showlegend=False), 1, 2)
fig.update_xaxes(title_text="rank by mutual information", row=1, col=2); fig.update_yaxes(title_text="rank by model importance", row=1, col=2)
fig.update_layout(template="plotly_white", height=560, title=f"Global explanation · {CHAMPION}")
fig.show()
record("explainability", method="permutation importance on validation rows; occlusion for single decisions", top_drivers=", ".join(gimp.head(5).index))

# 5 · Model testing
Everything in this section uses the **test set**, which no step above has touched.

In [ ]:
# 📊 Evaluation dashboard on the held-out test set
p_te = score_of(champion, Xte)
p_va = VAL_SCORES[CHAMPION]
TEST_SCORES = {m: score_of(FITTED[m], Xte) for m in ok.model.head(4) if m != CHAMPION}
TEST_SCORES[CHAMPION] = p_te
y = yte.to_numpy()
if TASK == "classification":
    grid = np.linspace(0.02, 0.98, 97)
    THRESHOLD = float(grid[np.argmax([skm.f1_score(yva, VAL_SCORES[CHAMPION] >= t) for t in grid])])          # chosen on validation
    pred = (p_te >= THRESHOLD).astype(int)
    perf = {"roc_auc": skm.roc_auc_score(y, p_te), "average_precision": skm.average_precision_score(y, p_te), "f1": skm.f1_score(y, pred),
            "precision": skm.precision_score(y, pred, zero_division=0), "recall": skm.recall_score(y, pred), "accuracy": skm.accuracy_score(y, pred),
            "balanced_accuracy": skm.balanced_accuracy_score(y, pred), "mcc": skm.matthews_corrcoef(y, pred), "brier": skm.brier_score_loss(y, p_te),
            "log_loss": skm.log_loss(y, np.clip(p_te, 1e-6, 1 - 1e-6)), "ks": float(np.max(np.abs(np.cumsum(y[np.argsort(-p_te)]) / y.sum() - np.cumsum(1 - y[np.argsort(-p_te)]) / (1 - y).sum()))),
            "gini": 2 * skm.roc_auc_score(y, p_te) - 1, "threshold": THRESHOLD}
    fig = make_subplots(rows=3, cols=3, vertical_spacing=0.1, horizontal_spacing=0.08,
                        subplot_titles=("ROC · all finalists", "Precision–recall · all finalists", "Calibration", "Score distribution by outcome",
                                        "Cumulative gains", "Lift by decile", f"Confusion matrix @ {THRESHOLD:.2f}", "KS: separation of the two classes", "Precision, recall, F1 by threshold"))
    for i, (m, s) in enumerate(TEST_SCORES.items()):
        fpr, tpr, _ = skm.roc_curve(y, s); pr, rc, _ = skm.precision_recall_curve(y, s)
        st = dict(color=BLUE if m == CHAMPION else PALETTE[(i + 1) % 8], width=3.5 if m == CHAMPION else 1.5)
        fig.add_trace(go.Scatter(x=fpr, y=tpr, name=f"{m} ({skm.roc_auc_score(y, s):.3f})", line=st), 1, 1)
        fig.add_trace(go.Scatter(x=rc, y=pr, line=st, showlegend=False), 1, 2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 1)
    cal = pd.DataFrame({"p": p_te, "y": y, "b": pd.qcut(p_te, 10, labels=False, duplicates="drop")}).groupby("b").mean()
    fig.add_trace(go.Scatter(x=cal.p, y=cal.y, mode="lines+markers", line=dict(color=TEAL, width=3), showlegend=False), 1, 3)
    fig.add_trace(go.Scatter(x=[0, cal.p.max()], y=[0, cal.p.max()], line=dict(color=GREY, dash="dot"), showlegend=False), 1, 3)
    for cls, col in ((0, BLUE), (1, RED)):
        fig.add_trace(go.Histogram(x=p_te[y == cls], nbinsx=40, opacity=0.6, marker_color=col, histnorm="probability", showlegend=False), 2, 1)
    order = np.argsort(-p_te); share = np.arange(1, len(y) + 1) / len(y); gains = np.cumsum(y[order]) / y.sum()
    fig.add_trace(go.Scatter(x=share, y=gains, line=dict(color=BLUE, width=3), showlegend=False), 2, 2)
    fig.add_trace(go.Scatter(x=[0, 1], y=[0, 1], line=dict(color=GREY, dash="dot"), showlegend=False), 2, 2)
    dec = pd.DataFrame({"d": pd.qcut(pd.Series(p_te).rank(method="first", ascending=False), 10, labels=False), "y": y}).groupby("d").y.mean() / y.mean()
    fig.add_trace(go.Bar(x=[f"D{i + 1}" for i in dec.index], y=dec.values, marker_color=ORANGE, showlegend=False), 2, 3)
    cm = skm.confusion_matrix(y, pred)
    fig.add_trace(go.Heatmap(z=cm[::-1], x=[f"pred {c}" for c in CLASS_NAMES], y=[f"true {c}" for c in CLASS_NAMES[::-1]], text=cm[::-1], texttemplate="%{text}", colorscale="Blues", showscale=False), 3, 1)
    cum1, cum0 = np.cumsum(y[order]) / y.sum(), np.cumsum(1 - y[order]) / (1 - y).sum()
    fig.add_trace(go.Scatter(x=share, y=cum1, line=dict(color=RED), showlegend=False), 3, 2); fig.add_trace(go.Scatter(x=share, y=cum0, line=dict(color=BLUE), showlegend=False), 3, 2)
    fig.add_trace(go.Scatter(x=grid, y=[skm.precision_score(y, p_te >= t, zero_division=0) for t in grid], name="precision", line=dict(color=TEAL)), 3, 3)
    fig.add_trace(go.Scatter(x=grid, y=[skm.recall_score(y, p_te >= t) for t in grid], name="recall", line=dict(color=ORANGE)), 3, 3)
    fig.add_trace(go.Scatter(x=grid, y=[skm.f1_score(y, p_te >= t) for t in grid], name="F1", line=dict(color=DARK, width=3)), 3, 3)
    fig.add_vline(x=THRESHOLD, line_dash="dot", line_color=RED, row=3, col=3)
    fig.update_layout(barmode="overlay")
else:
    res = y - p_te
    perf = {"r2": skm.r2_score(y, p_te), "rmse": float(np.sqrt(np.mean(res ** 2))), "mae": float(np.mean(np.abs(res))), "median_abs_error": float(np.median(np.abs(res))),
            "mape_pct": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9)) * 100), "explained_variance": skm.explained_variance_score(y, p_te),
            "max_error": float(np.max(np.abs(res))), "within_20pct_share": float(np.mean(np.abs(res) / np.maximum(np.abs(y), 1e-9) <= 0.2) * 100)}
    ape = np.abs(res) / np.maximum(np.abs(y), 1e-9) * 100
    fig = make_subplots(rows=2, cols=3, vertical_spacing=0.14, subplot_titles=("Predicted vs actual", "Residuals vs prediction", "Residual distribution",
                                                                               "Median % error by actual-value decile", "Share of predictions within an error band", "R² of the finalists"))
    lo, hi = np.percentile(np.r_[y, p_te], [1, 99])
    fig.add_trace(go.Scattergl(x=y, y=p_te, mode="markers", marker=dict(size=4, color=BLUE, opacity=0.4), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=[lo, hi], y=[lo, hi], line=dict(color=DARK, dash="dot"), showlegend=False), 1, 1)
    fig.add_trace(go.Scattergl(x=p_te, y=res, mode="markers", marker=dict(size=4, color=ORANGE, opacity=0.4), showlegend=False), 1, 2)
    fig.add_trace(go.Histogram(x=res, nbinsx=60, marker_color=GREY, showlegend=False), 1, 3)
    bd = pd.DataFrame({"d": pd.qcut(y, 10, labels=False, duplicates="drop"), "ape": ape}).groupby("d").ape.median()
    fig.add_trace(go.Bar(x=[f"D{int(i) + 1}" for i in bd.index], y=bd.values, marker_color=TEAL, showlegend=False), 2, 1)
    bands = [10, 20, 35, 50]
    fig.add_trace(go.Bar(x=[f"within {b}%" for b in bands], y=[float(np.mean(ape <= b) * 100) for b in bands], marker_color=BLUE, showlegend=False,
                         text=[f"{np.mean(ape <= b) * 100:.0f}%" for b in bands], textposition="outside"), 2, 2)
    r2s = {m: skm.r2_score(y, s) for m, s in TEST_SCORES.items()}
    fig.add_trace(go.Bar(x=list(r2s), y=list(r2s.values()), marker_color=[BLUE if m == CHAMPION else GREY for m in r2s], showlegend=False,
                         text=[f"{v:.3f}" for v in r2s.values()], textposition="outside"), 2, 3)
fig.update_layout(template="plotly_white", height=980 if TASK == "classification" else 760, title=f"Test-set evaluation · {CHAMPION} · {len(y):,} unseen rows",
                  legend=dict(orientation="h", y=-0.06))
fig.show()
kp = list(perf.items())[:6]
fig = make_subplots(rows=1, cols=len(kp), specs=[[{"type": "indicator"}] * len(kp)])
for i, (k, v) in enumerate(kp, 1):
    fig.add_trace(go.Indicator(mode="number", value=v, title={"text": k.replace("_", " ")}, number={"valueformat": ",.3f" if abs(v) < 100 else ",.0f", "font": {"size": 40}}), 1, i)
fig.update_layout(height=200, margin=dict(t=50, b=10)); fig.show()
record("performance", evaluated_on="held-out test set", test_rows=len(y), **perf)

In [ ]:
# 🎚️ Interactive: move the decision threshold (classification) or the tolerance (regression)
def explore(value):
    if TASK == "classification":
        pr = (p_te >= value).astype(int); cm = skm.confusion_matrix(y, pr)
        txt = (f"threshold {value:.2f} · flagged {pr.mean():.1%} of applications · precision {skm.precision_score(y, pr, zero_division=0):.3f} · "
               f"recall {skm.recall_score(y, pr):.3f} · F1 {skm.f1_score(y, pr):.3f}")
        f = go.Figure(go.Heatmap(z=cm[::-1], x=[f"pred {c}" for c in CLASS_NAMES], y=[f"true {c}" for c in CLASS_NAMES[::-1]], text=cm[::-1],
                                 texttemplate="%{text}", colorscale="Blues", showscale=False))
    else:
        tol = value; a = np.abs(y - p_te) / np.maximum(np.abs(y), 1e-9) * 100
        txt = f"tolerance ±{tol:.0f}% · {np.mean(a <= tol):.1%} of recommendations are inside it"
        f = go.Figure(go.Histogram(x=np.clip(a, 0, 150), nbinsx=60, marker_color=[BLUE]))
        f.add_vline(x=tol, line_color=RED, line_dash="dot")
    f.update_layout(template="plotly_white", height=380, title=txt); f.show()

slider = W.FloatSlider(value=THRESHOLD, min=0.05, max=0.95, step=0.01, description="threshold") if TASK == "classification" \
    else W.FloatSlider(value=20, min=5, max=60, step=1, description="± % error")
W.interact(explore, value=slider);

In [ ]:
# 🧾 Per-decision explanations (occlusion): replace one feature with a typical value and see how the prediction moves
TYPICAL = {f: (Xtr[f].median() if f not in CAT else Xtr[f].mode().iloc[0]) for f in FEATURES}
LOCAL_FEATURES = gimp.head(6 if FAST else 12).index.tolist()

def explain_case(row):
    variants = pd.concat([row.to_frame().T] * (len(LOCAL_FEATURES) + 1), ignore_index=True).astype(Xte.dtypes.to_dict())
    for i, f in enumerate(LOCAL_FEATURES, 1):
        variants.loc[i, f] = TYPICAL[f]
    s = score_of(champion, variants)
    return float(s[0]), pd.Series(s[0] - s[1:], index=LOCAL_FEATURES)

order_ = np.argsort(-p_te)
picks = {"highest prediction": order_[0], "second highest": order_[1], "median": order_[len(order_) // 2], "lowest prediction": order_[-1]}
fig, n_tr = go.Figure(), 0
for k, (label, i) in enumerate(picks.items()):
    base_, contrib = explain_case(Xte.iloc[i])
    c = contrib.reindex(contrib.abs().sort_values().index)
    fig.add_trace(go.Bar(x=c.values, y=[f"{f[:26]} = {Xte.iloc[i][f]:.3g}" if f not in CAT else f"{f[:22]} = {Xte.iloc[i][f]}" for f in c.index], orientation="h",
                         marker_color=[RED if v > 0 else BLUE for v in c.values], visible=k == 0,
                         name=f"{label}: prediction {base_:.3f}, actual {y[i]:.3g}"))
fig.update_layout(template="plotly_white", height=520, showlegend=False,
                  title="Why this prediction? Red pushed it up, blue pushed it down (vs a typical applicant)",
                  updatemenus=[dict(x=1, xanchor="right", y=1.18, buttons=[dict(label=l, method="update", args=[{"visible": [j == k for j in range(len(picks))]}])
                                                                               for k, l in enumerate(picks)])])
fig.show()

# 6 · Model inference in batch mode
A file of new applications is scored in chunks, decisions and uncertainty are attached, and the result is written out. The fitted pipeline is saved and reloaded first, to prove the artefact is self-contained.

In [ ]:
# 📦 Save → reload → score a batch file → write results
ARTEFACT, BATCH_IN, BATCH_OUT = "champion_pipeline.joblib", "batch_input.csv", "batch_scored.csv"
scorer = champion
if hasattr(champion, "save"):
    try:
        champion.save(ARTEFACT); scorer = TabularPipeline.load(ARTEFACT)
        print(f"saved and reloaded {ARTEFACT} ({os.path.getsize(ARTEFACT) / 1e6:.1f} MB)")
    except Exception as exc:
        print("save/load skipped:", str(exc)[:100])
batch = stream.head(N_MONTH)[[UID] + FEATURES]
batch.to_csv(BATCH_IN, index=False)
incoming = pd.read_csv(BATCH_IN).astype(batch.dtypes.to_dict())
chunks, timings, out = np.array_split(np.arange(len(incoming)), 5), [], []
for ch in chunks:
    t0 = time.time(); s = score_of(scorer, incoming.iloc[ch][FEATURES]); timings.append(len(ch) / (time.time() - t0)); out.append(s)
scored = incoming[[UID]].assign(prediction=np.concatenate(out))
if TASK == "classification":
    scored["decision"] = np.where(scored.prediction >= THRESHOLD, "review / decline", "approve")
scored.to_csv(BATCH_OUT, index=False)
fig = make_subplots(rows=1, cols=2, subplot_titles=("Throughput per chunk (rows/second)", "Distribution of batch predictions"))
fig.add_trace(go.Bar(x=[f"chunk {i + 1}" for i in range(len(timings))], y=timings, marker_color=TEAL, showlegend=False), 1, 1)
fig.add_trace(go.Histogram(x=scored.prediction, nbinsx=40, marker_color=BLUE, showlegend=False), 1, 2)
fig.update_layout(template="plotly_white", height=380, title=f"Batch scoring · {len(scored):,} rows → {BATCH_OUT}")
fig.show()
display(scored.head(8))
record("performance", batch_rows_per_second=float(np.median(timings)))
display(Markdown("**On the Lexsi platform** the same step is `project.upload_data(\"batch.csv\", \"Batch1\")` followed by "
                 "`project.model_inference(tag=\"Batch1\", model_name=..., compute_type=\"T4.small\")`, which returns the scored tag."))

# 7 · Model risk

## 7.1 Uncertainty
A single number hides how sure the model is. **Conformal prediction** turns each prediction into a set (classification) or an interval (regression) with a coverage promise that is then measured on the test set.

In [ ]:
# 🎯 Conformal prediction and calibration
from tabtune.uncertainty import ConformalClassifier, ConformalRegressor, Recalibrator
ALPHA = 0.10
if TASK == "classification":
    cp = ConformalClassifier(champion, method="lac", alpha=ALPHA).calibrate(Xva, yva)
    cov = cp.coverage(Xte, yte); sizes_ = np.asarray(cp.set_sizes(Xte))
    def ece(p, yy, bins=10):
        b = np.minimum((p * bins).astype(int), bins - 1)
        return float(sum(abs(p[b == k].mean() - yy[b == k].mean()) * (b == k).mean() for k in range(bins) if (b == k).any()))
    p_recal = p_te
    try:
        rc = Recalibrator(champion, method="temperature").fit(Xva, yva); p_recal = np.asarray(rc.predict_proba(Xte))[:, 1]
    except Exception as exc:
        print("recalibration skipped:", str(exc)[:100])
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(cov["coverage"]), "average_set_size": float(sizes_.mean()),
           "share_single_answer": float((sizes_ == 1).mean()), "share_needs_review": float((sizes_ != 1).mean()), "ece_before": ece(p_te, y), "ece_after_recalibration": ece(p_recal, y)}
    fig = make_subplots(rows=1, cols=3, subplot_titles=("Coverage: promised vs measured", "Prediction-set size", "Calibration error before and after"))
    fig.add_trace(go.Bar(x=["promised", "measured"], y=[1 - ALPHA, unc["measured_coverage"]], marker_color=[GREY, TEAL], text=[f"{1 - ALPHA:.0%}", f"{unc['measured_coverage']:.1%}"], textposition="outside", showlegend=False), 1, 1)
    vc = pd.Series(sizes_).value_counts().sort_index()
    fig.add_trace(go.Bar(x=[{0: "abstain", 1: "one answer", 2: "both possible"}.get(int(k), str(k)) for k in vc.index], y=vc.values, marker_color=[RED, BLUE, ORANGE][:len(vc)], showlegend=False), 1, 2)
    fig.add_trace(go.Bar(x=["raw", "temperature-scaled"], y=[unc["ece_before"], unc["ece_after_recalibration"]], marker_color=[GREY, TEAL], showlegend=False), 1, 3)
else:
    cp = ConformalRegressor(champion, method="absolute", alpha=ALPHA).calibrate(Xva, yva)
    lo_, hi_ = cp.predict_interval(Xte); lo_, hi_ = np.asarray(lo_), np.asarray(hi_); cov = cp.coverage(Xte, yte)
    unc = {"target_coverage": 1 - ALPHA, "measured_coverage": float(cov["coverage"]), "average_interval_width": float(np.mean(hi_ - lo_)),
           "interval_width_vs_median_target_pct": float(np.mean(hi_ - lo_) / np.median(y) * 100)}
    o = np.argsort(p_te)[:: max(1, len(y) // 150)]
    fig = make_subplots(rows=1, cols=2, column_widths=[0.3, 0.7], subplot_titles=("Coverage: promised vs measured", "Intervals for a sample of applicants (sorted by prediction)"))
    fig.add_trace(go.Bar(x=["promised", "measured"], y=[1 - ALPHA, unc["measured_coverage"]], marker_color=[GREY, TEAL], text=[f"{1 - ALPHA:.0%}", f"{unc['measured_coverage']:.1%}"], textposition="outside", showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=np.r_[np.arange(len(o)), np.arange(len(o))[::-1]], y=np.r_[hi_[o], lo_[o][::-1]], fill="toself", fillcolor="rgba(47,91,234,0.18)", line=dict(width=0), name="90% interval"), 1, 2)
    fig.add_trace(go.Scatter(x=np.arange(len(o)), y=p_te[o], line=dict(color=BLUE), name="prediction"), 1, 2)
    fig.add_trace(go.Scatter(x=np.arange(len(o)), y=y[o], mode="markers", marker=dict(color=ORANGE, size=5), name="actual"), 1, 2)
fig.update_layout(template="plotly_white", height=420, title="Uncertainty you can promise: split conformal prediction, checked on the test set")
fig.show()
record("uncertainty", method="split conformal prediction calibrated on the validation set", **unc)

## 7.2 Fairness

In [ ]:
# ⚖️ Group fairness on the test set
gcols = [c for c in test.columns if c.startswith(GROUP_PREFIX)]
if len(gcols) > 1:
    group = test[gcols].idxmax(axis=1).str.replace(GROUP_PREFIX, "")
elif gcols:
    group = test[gcols[0]].astype(str)
else:
    group = pd.Series("all", index=test.index)
group = group.where(group.map(group.value_counts()) >= 30, "other")
if TASK == "classification":
    g = pd.DataFrame({"g": group.values, "y": y, "flag": (p_te >= THRESHOLD).astype(int)})
    tab = g.groupby("g").apply(lambda d: pd.Series({"rows": len(d), "base_rate": d.y.mean(), "selection_rate": d.flag.mean(),
                                                    "true_positive_rate": d.flag[d.y == 1].mean(), "false_positive_rate": d.flag[d.y == 0].mean(),
                                                    "precision": d.y[d.flag == 1].mean() if d.flag.sum() else np.nan}))
    fair = {"demographic_parity_difference": float(tab.selection_rate.max() - tab.selection_rate.min()),
            "disparate_impact_ratio": float(tab.selection_rate.min() / max(tab.selection_rate.max(), 1e-9)),
            "equal_opportunity_difference": float(tab.true_positive_rate.max() - tab.true_positive_rate.min()),
            "false_positive_rate_difference": float(tab.false_positive_rate.max() - tab.false_positive_rate.min()),
            "equalized_odds_difference": float(max(tab.true_positive_rate.max() - tab.true_positive_rate.min(), tab.false_positive_rate.max() - tab.false_positive_rate.min())),
            "predictive_parity_difference": float(tab.precision.max() - tab.precision.min())}
    show = ["selection_rate", "true_positive_rate", "false_positive_rate", "precision"]
else:
    g = pd.DataFrame({"g": group.values, "y": y, "p": p_te})
    tab = g.groupby("g").apply(lambda d: pd.Series({"rows": len(d), "mean_actual": d.y.mean(), "mean_prediction": d.p.mean(),
                                                    "mae": (d.y - d.p).abs().mean(), "mean_error": (d.p - d.y).mean(), "mape_pct": ((d.y - d.p).abs() / d.y.abs().clip(lower=1e-9)).mean() * 100}))
    fair = {"mean_prediction_ratio": float(tab.mean_prediction.min() / tab.mean_prediction.max()), "mae_ratio": float(tab.mae.min() / tab.mae.max()),
            "mean_error_gap": float(tab.mean_error.max() - tab.mean_error.min()), "mape_gap_pct": float(tab.mape_pct.max() - tab.mape_pct.min()),
            "calibration_gap": float((tab.mean_prediction / tab.mean_actual).max() - (tab.mean_prediction / tab.mean_actual).min())}
    show = ["mean_actual", "mean_prediction", "mae", "mape_pct"]
fig = make_subplots(rows=1, cols=2, column_widths=[0.55, 0.45], horizontal_spacing=0.24, subplot_titles=(f"Rates by {GROUP_LABEL}", "Fairness metrics (0 = no gap; ratio: 1 = equal)"))
for i, m in enumerate(show):
    v = tab[m] / (tab[m].max() if TASK == "regression" else 1)
    fig.add_trace(go.Bar(x=tab.index, y=v, name=m.replace("_", " ") + (" (scaled)" if TASK == "regression" else ""), marker_color=PALETTE[i]), 1, 1)
fig.add_trace(go.Bar(x=list(fair.values()), y=[k.replace("_", " ") for k in fair], orientation="h", marker_color=GREY, showlegend=False,
                     text=[f"{v:.3f}" for v in fair.values()], textposition="outside"), 1, 2)
fig.update_xaxes(range=[0, max(fair.values()) * 1.25], row=1, col=2)
fig.update_layout(template="plotly_white", height=460, barmode="group", title=f"Fairness by {GROUP_LABEL} · {len(tab)} groups", legend=dict(orientation="h", y=-0.2))
fig.show()
display(tab.style.format(precision=3))
record("fairness", attribute=GROUP_LABEL, groups=len(tab), **fair)
record("limitations", note_1="fairness is measured on one attribute available in the data; other protected attributes were not available",
       note_2="zero-shot foundation models depend on the context rows; refresh the context when the population shifts",
       note_3="results come from a sample of the source data, not the full portfolio")

## 7.3 Regulatory evidence map, and accuracy checks
The table links each model-risk expectation to the evidence this notebook produced. The second cell runs sanity checks on the predictions themselves; the model card's own accuracy is checked in section 8.

In [ ]:
# 🏛️ Model risk: which evidence in this notebook supports which expectation
REG_MAP = pd.DataFrame([
    ("Conceptual soundness", "Problem statement, data preparation log, feature rationale, model selection rule", "SR 11-7 · SAMA model risk expectations"),
    ("Data quality and lineage", "Data profile, agent action log with human approvals, leakage checks", "SAMA Data Management · NDMO standards · PDPL"),
    ("Outcomes analysis", "Hold-out metrics, evaluation dashboards, segment performance", "SR 11-7 · Basel sound-practice principles"),
    ("Uncertainty", "Conformal prediction with measured coverage, calibration", "SR 11-7 (model limitations) · CBB AI policy"),
    ("Fairness", "Group metrics on protected or proxy attributes", "SDAIA AI Ethics Principles · CBB AI policy · EU AI Act"),
    ("Explainability", "Global drivers and per-decision explanations", "SDAIA AI Ethics Principles · EU AI Act"),
    ("Ongoing monitoring", "Drift monitoring, performance tracking, retraining with approval gate", "SR 11-7 · SAMA model risk expectations"),
    ("Documentation", "Generated model card with grounding check, audit trail", "SR 11-7 · ISO/IEC 42001"),
], columns=["Expectation", "Evidence produced in this notebook", "Frameworks it speaks to"])
display(REG_MAP.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"})
        .set_caption("Frameworks are named for orientation only. The bank's Compliance and Risk functions own the control mapping; "
                     "no clause-level compliance is claimed here."))

In [ ]:
# 🧪 Prediction sanity checks (automatic accuracy guards)
checks = [("no missing predictions", bool(np.isfinite(p_te).all())),
          ("predictions in a valid range", bool(((p_te >= 0) & (p_te <= 1)).all()) if TASK == "classification" else bool((p_te > 0).mean() > 0.99)),
          ("deterministic: same rows give the same answer", bool(np.allclose(score_of(champion, Xte.head(40)), p_te[:40], rtol=1e-3, atol=1e-3))),
          ("beats the linear baseline on validation", bool(ok.set_index("model").loc[CHAMPION, PRIMARY] >= ok.set_index("model")[PRIMARY].get("Linear baseline", -np.inf))),
          ("test score close to validation score (no selection overfit)", bool(abs(metric(y, p_te) - ok.set_index("model").loc[CHAMPION, PRIMARY]) < 0.05)),
          ("conformal coverage within 3 points of the promise", bool(abs(unc["measured_coverage"] - (1 - ALPHA)) < 0.03))]
display(pd.DataFrame(checks, columns=["check", "passed"]).style.hide(axis="index").map(lambda v: f"color:{TEAL if v else RED};font-weight:600", subset=["passed"]))
record("governance", frameworks_referenced="SR 11-7; SAMA model risk expectations; SAMA and NDMO data standards; PDPL; CBB AI policy; SDAIA AI Ethics Principles; EU AI Act; ISO/IEC 42001",
       sanity_checks_passed=int(sum(v for _, v in checks)), sanity_checks_total=len(checks), human_approval_gates=sum(1 for a in AUDIT if a["actor"] == "human"))

# 8 · Artefacts before production

## 8.1 Model card
An open-source LLM (gpt-oss-120b (OpenAI open weights, Apache-2.0), running on this notebook's GPU) writes the model card **only from the facts recorded above**. The cell shows the trace (model, tokens, time) and then checks every number in the card against the fact sheet: that is the hallucination check. On the Lexsi platform this is the job of the Attest agent.

**Writer model.** `openai/gpt-oss-120b` is a 117-billion-parameter mixture-of-experts model (about 5B parameters active per token) whose weights ship in 4-bit MXFP4, about 63 GB, sized to run on one 80 GB H100. The first run downloads about 65 GB (roughly 5–10 minutes). It thinks in an *analysis* channel before writing its *final* answer; the reasoning is shown below the trace table. If it cannot load, the cell falls back to Qwen2.5-32B-Instruct, then Qwen2.5-7B, then the template writer, and the trace says which one wrote the card.

In [ ]:
# 📝 Model card written by an open-source LLM from the recorded facts, with a full trace and a grounding check
LLM_CANDIDATES = ['openai/gpt-oss-120b', 'Qwen/Qwen2.5-32B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']   # tried in order; set [] to use the template writer
LLM_MODEL = LLM_CANDIDATES[0] if LLM_CANDIDATES else None
USE_LLM = DEVICE == "cuda" and not FAST and bool(LLM_CANDIDATES)
_LLM = {}


def _load_llm():
    """Load the first candidate that fits; frees GPU memory between attempts."""
    import gc
    from transformers import AutoModelForCausalLM, AutoTokenizer
    errors = []
    for name in LLM_CANDIDATES:
        try:
            t0 = time.time()
            gc.collect(); torch.cuda.empty_cache()
            tok = AutoTokenizer.from_pretrained(name)
            # gpt-oss ships MXFP4 weights: dtype "auto" keeps them 4-bit (about 63 GB for the 120b model on one 80 GB GPU)
            model = AutoModelForCausalLM.from_pretrained(name, torch_dtype="auto" if "gpt-oss" in name else torch.bfloat16, device_map="auto")
            _LLM.update(tok=tok, model=model, name=name, load_seconds=round(time.time() - t0, 1),
                        gpu_gb=round(torch.cuda.memory_allocated() / 1e9, 1))
            print(f"loaded {name} in {_LLM['load_seconds']}s · GPU memory in use {_LLM['gpu_gb']} GB")
            return
        except Exception as exc:
            errors.append(f"{name}: {str(exc)[:120]}")
            print(f"could not load {name} → trying the next option ({str(exc)[:120]})")
            for k in ("tok", "model"):
                _LLM.pop(k, None)
            gc.collect(); torch.cuda.empty_cache()
    raise RuntimeError("no LLM candidate could be loaded: " + " | ".join(errors))


def _split_harmony(text):
    """gpt-oss answers in two channels: 'analysis' (its reasoning) and 'final' (the answer)."""
    final = re.search(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)", text, flags=re.S)
    analysis = re.search(r"<\|channel\|>analysis<\|message\|>(.*?)(?:<\|end\|>|<\|start\|>|$)", text, flags=re.S)
    if final:
        return final.group(1).strip(), (analysis.group(1).strip() if analysis else "")
    return re.sub(r"<\|[^|]+\|>", "", text).strip(), ""
CARD_SECTIONS = ["1. Model overview and intended use", "2. Data and preparation", "3. Model selection and training",
                 "4. Performance on held-out data", "5. Explainability", "6. Uncertainty, fairness and limitations",
                 "7. Monitoring and retraining plan", "8. Governance and regulatory alignment", "9. Change log"]
SYSTEM_PROMPT = ("You are a model-risk documentation writer for a bank regulated by SAMA and the CBB. Write a model card in Markdown with "
                 "exactly these sections: " + "; ".join(CARD_SECTIONS) + ". Use ONLY the facts in the JSON the user gives you. "
                 "Quote numbers exactly as given. If a fact is missing, write 'not recorded'. Do not invent figures, dates, clause numbers or approvals. "
                 "In section 8 name the frameworks listed in the facts and state that control mapping is owned by the bank's Compliance function.")


def _json_safe(o):
    if isinstance(o, dict):
        return {str(k): _json_safe(v) for k, v in o.items()}
    if isinstance(o, (list, tuple, set)):
        return [_json_safe(v) for v in o]
    if isinstance(o, (np.floating, np.integer)):
        return float(o)
    if isinstance(o, (pd.Timestamp,)):
        return str(o)
    return o


def _template_card(facts, changes):
    """Deterministic writer used when no LLM is available: every line comes straight from the facts."""
    lines = [f"# Model card · {facts.get('use_case', {}).get('name', 'model')}", "", "*Written by the template writer (no LLM was run).*", ""]
    keys = list(facts)
    alloc = {0: ["use_case"], 1: ["data", "preparation", "features"], 2: ["selection", "training"], 3: ["performance"],
             4: ["explainability"], 5: ["uncertainty", "fairness", "limitations"], 6: ["monitoring", "retraining"], 7: ["governance"]}
    for i, title in enumerate(CARD_SECTIONS[:8]):
        lines += [f"## {title}"]
        rows = [(k, v) for sec in alloc[i] if sec in facts for k, v in facts[sec].items()]
        lines += [f"- **{k.replace('_', ' ')}:** {v}" for k, v in rows] or ["- not recorded"]
        lines.append("")
    lines += [f"## {CARD_SECTIONS[8]}"] + ([f"- {c}" for c in changes] if changes else ["- Initial version."])
    return "\n".join(lines)


def generate_model_card(facts, changes=None, version_label="v1"):
    facts = _json_safe(facts)
    user = "FACTS (JSON):\n" + json.dumps(facts, indent=1) + ("\n\nCHANGES SINCE THE PREVIOUS VERSION:\n- " + "\n- ".join(changes) if changes else "")
    trace = {"version": version_label, "writer": LLM_MODEL if USE_LLM else "template writer", "prompt_characters": len(SYSTEM_PROMPT) + len(user)}
    t0 = time.time()
    if USE_LLM:
        try:
            if "model" not in _LLM:
                _load_llm()
            tok, model, name = _LLM["tok"], _LLM["model"], _LLM["name"]
            msgs = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}]
            harmony = "gpt-oss" in name
            kw = {"reasoning_effort": "low"} if harmony else {}
            ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", **kw).to(model.device)
            out = model.generate(ids, max_new_tokens=3500 if harmony else 1600, do_sample=False)
            raw = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=not harmony)
            card, reasoning = _split_harmony(raw) if harmony else (raw.strip(), "")
            n_out = int(out.shape[1] - ids.shape[1])
            trace.update(writer=name, input_tokens=int(ids.shape[1]), output_tokens=n_out, load_seconds=_LLM["load_seconds"],
                         gpu_memory_gb=_LLM["gpu_gb"], reasoning_characters=len(reasoning))
            trace["_reasoning"] = reasoning
        except Exception as exc:
            trace["writer"] = f"template writer (LLM failed: {str(exc)[:80]})"
            card = _template_card(facts, changes)
    else:
        card = _template_card(facts, changes)
    trace["seconds"] = round(time.time() - t0, 1)
    if trace.get("output_tokens"):
        trace["tokens_per_second"] = round(trace["output_tokens"] / max(trace["seconds"], 1e-6), 1)

    # Grounding check: every number in the card must appear in the facts (or the change list)
    source = json.dumps(facts) + " " + " ".join(changes or [])
    known = {n.rstrip("0").rstrip(".") if "." in n else n for n in re.findall(r"\d+(?:\.\d+)?", source.replace(",", ""))}
    found = re.findall(r"\d+(?:\.\d+)?", re.sub(r"^#+ \d+\.", "", card.replace(",", ""), flags=re.M))
    norm = lambda n: n.rstrip("0").rstrip(".") if "." in n else n
    checked = [n for n in found if not (n.isdigit() and int(n) < 13)]              # skip list numbering and small counts
    unverified = sorted({n for n in checked if norm(n) not in known})
    missing = [s for s in CARD_SECTIONS if s.split(". ", 1)[1].split(" ")[0].lower() not in card.lower()]
    trace.update(numbers_checked=len(checked), numbers_not_in_facts=len(unverified),
                 grounded_pct=round(100 * (1 - len(unverified) / max(len(checked), 1)), 1), sections_missing=len(missing))
    return card, trace, unverified


def show_card(card, trace, unverified):
    reasoning = trace.get("_reasoning", "")
    display(pd.DataFrame([{k: v for k, v in trace.items() if not k.startswith("_")}]).T.rename(columns={0: "trace"}))
    if reasoning:
        import html as _html
        display(HTML(f"<details><summary><b>LLM reasoning trace</b> (analysis channel, {len(reasoning):,} characters) · click to expand</summary>"
                     f"<pre style='white-space:pre-wrap;font-size:12px'>{_html.escape(reasoning[:8000])}</pre></details>"))
    fig = go.Figure(go.Indicator(mode="gauge+number", value=trace["grounded_pct"], number={"suffix": "%"},
                                 title={"text": "Numbers in the card that trace back to recorded facts"},
                                 gauge={"axis": {"range": [0, 100]}, "bar": {"color": TEAL if trace["grounded_pct"] >= 99 else RED}}))
    fig.update_layout(height=260, margin=dict(t=60, b=10)); fig.show()
    if unverified:
        print("numbers NOT found in the facts (review before use):", unverified[:20])
    display(Markdown(card))

In [ ]:
# 📝 Generate model card v1
for _m in [m for m in list(FITTED) if m != CHAMPION]:           # free GPU memory held by the losing candidates before loading the LLM
    FITTED.pop(_m, None)
import gc; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()
record("use_case", name="Home Credit · Loan Amount Recommendation", task=TASK, decision_supported="recommended credit amount with a range", owner="[model owner]", version="v1")
record("monitoring", plan="monthly drift check (PSI) and performance check; retraining proposed by an agent and approved by a human")
CARD_V1, TRACE_V1, UNVERIFIED_V1 = generate_model_card(FACTS, version_label="v1")
show_card(CARD_V1, TRACE_V1, UNVERIFIED_V1)
with open("model_card_v1.md", "w") as f:
    f.write(CARD_V1)

## 8.2 Production API contract
The request follows the Lexsi case-registration call (`register_case` in the Lexsi SDK: project, unique identifier, tag, data, compute and explanation methods). The cell validates a real request against the schema and answers it with the selected model, so the contract is tested, not just written.

In [ ]:
# 🔌 API contract: schema, validation, and a live round trip with the selected model
SCHEMA = {f: ("string" if f in CAT else "number") for f in FEATURES}
def score_api(request):
    t0 = time.time()
    missing_ = [f for f in FEATURES if f not in request["data"][0]]
    if missing_:
        return {"success": False, "details": f"missing fields: {missing_[:5]}"}
    row = pd.DataFrame(request["data"])[FEATURES].astype(Xte.dtypes.to_dict())
    s = float(score_of(champion, row)[0]); _, contrib = explain_case(row.iloc[0])
    body = {"success": True, "model_name": CHAMPION, "model_version": "v1", "unique_identifier": request["unique_identifier"], "pred_value": round(s, 4)}
    if TASK == "classification":
        body.update(pred_category=CLASS_NAMES[int(s >= THRESHOLD)], decision="review / decline" if s >= THRESHOLD else "approve", threshold=round(THRESHOLD, 2))
    else:
        lo1, hi1 = cp.predict_interval(row); body.update(interval_90=[round(float(np.asarray(lo1)[0]), 2), round(float(np.asarray(hi1)[0]), 2)])
    body.update(top_drivers={k: round(float(v), 4) for k, v in contrib.reindex(contrib.abs().sort_values(ascending=False).index).head(5).items()},
                audit_trail={"latency_ms": round((time.time() - t0) * 1000), "explanation_method": "occlusion", "scored_at": pd.Timestamp.now().isoformat(timespec="seconds")})
    return body

i = int(order_[0])
request = {"project_name": "[lexsi project]", "unique_identifier": str(test.iloc[i][UID]), "tag": "Live", "serverless_type": "nova-1", "xai": ["shap"],
           "data": [json.loads(Xte.iloc[[i]].to_json(orient="records"))[0]]}
response = score_api(request)
print("POST /case   headers: x-api-token\n\nREQUEST (first 6 features shown)")
print(json.dumps({**request, "data": [dict(list(request["data"][0].items())[:6])]}, indent=1))
print("\nRESPONSE"); print(json.dumps(response, indent=1))
display(pd.DataFrame({"field": list(SCHEMA)[:12], "type": list(SCHEMA.values())[:12]}).T.style.set_caption(f"Input schema · {len(SCHEMA)} fields (first 12 shown)"))
print("rejects a bad request:", score_api({"unique_identifier": "x", "data": [{"wrong": 1}]})["details"][:60])

# 9 · Model in production

## 9.1 Drift monitoring
The production stream is replayed as monthly batches. To make the monitor earn its keep, the mix of applicants shifts a little more each month on the strongest feature (real rows, resampled; no values are altered).

In [ ]:
# 📡 Monthly monitoring: population stability (PSI) and performance of the model in production
def psi(ref, new, bins=10):
    ref, new = pd.Series(ref).dropna(), pd.Series(new).dropna()
    edges = np.unique(np.quantile(ref, np.linspace(0, 1, bins + 1)))
    if len(edges) < 3:
        return 0.0
    edges[0], edges[-1] = -np.inf, np.inf
    a = np.histogram(ref, edges)[0] / len(ref) + 1e-4; b = np.histogram(new, edges)[0] / max(len(new), 1) + 1e-4
    return float(np.sum((b - a) * np.log(b / a)))

drift_feature = next(c for c in gimp.index if c in cont)
z = ((stream[drift_feature] - stream[drift_feature].mean()) / (stream[drift_feature].std() + 1e-9)).fillna(0).clip(-3, 3)
pool, months = stream.copy(), []
for m in range(MONTHS):
    strength = 0.35 * m
    pick = pool.sample(min(N_MONTH, len(pool)), weights=np.exp(strength * z.loc[pool.index]), random_state=m)
    pool = pool.drop(pick.index); months.append(pick.reset_index(drop=True))
MON_FEATURES = [c for c in gimp.index if c in NUM][:12]
psi_tab = pd.DataFrame({f"M{m + 1}": {f: psi(Xtr[f], mo[f]) for f in MON_FEATURES} for m, mo in enumerate(months)})
month_scores = [score_of(champion, mo[FEATURES]) for mo in months]
perf_m = [metric(mo[TARGET_COL], s) for mo, s in zip(months, month_scores)]
pred_psi = [psi(p_va, s) for s in month_scores]
baseline_perf = metric(y, p_te)
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, specs=[[{"colspan": 2}, None], [{}, {}]],
                    subplot_titles=("Population stability index by feature and month (above 0.2 = material drift)", f"{PRIMARY} in production vs at release", "Drift of the model's own predictions (PSI)"))
fig.add_trace(go.Heatmap(z=psi_tab.values, x=psi_tab.columns, y=[c[:30] for c in psi_tab.index], colorscale=[[0, "#F7F7F4"], [0.4, GOLD], [1, RED]], zmin=0, zmax=0.5,
                         text=psi_tab.round(2).values, texttemplate="%{text}", colorbar=dict(title="PSI", len=0.45, y=0.8)), 1, 1)
fig.add_trace(go.Scatter(x=psi_tab.columns, y=perf_m, mode="lines+markers", line=dict(color=BLUE, width=3), showlegend=False), 2, 1)
fig.add_hline(y=baseline_perf, line_dash="dot", line_color=GREY, row=2, col=1)
fig.add_trace(go.Bar(x=psi_tab.columns, y=pred_psi, marker_color=[RED if v > 0.2 else GOLD if v > 0.1 else GREY for v in pred_psi], showlegend=False), 2, 2)
fig.update_layout(template="plotly_white", height=760, title=f"Production monitoring · {MONTHS} months · drifting feature: {drift_feature}")
fig.show()

## 9.2 Retraining agent with a human in the loop
Each month the agent **observes** drift and performance, **decides** whether retraining is warranted, asks a **human to approve**, **retrains** a challenger with the newest labelled data, **verifies** it on the following month, and **promotes** it only if it wins and a human signs off.

In [ ]:
# 🔁 Retraining agent: observe → decide → approve → retrain → verify → approve → promote
PSI_ALERT, DECAY_ALERT, PROMOTE_MARGIN = 0.20, 0.02, 0.002
state = {"model": champion, "name": CHAMPION, "version": 1, "reference": ctx, "events": [], "history": []}

def refit_like_champion(X, y_):
    """Retraining keeps the selected recipe and refreshes what it learns from."""
    if CHAMPION in ROSTER:
        return fit_tfm(CHAMPION, X, y_)
    if CHAMPION.endswith("+ LoRA"):
        return fit_tfm(CHAMPION.replace(" + LoRA", ""), X, y_, strategy="peft", tuning_params={"epochs": LORA_EPOCHS, "learning_rate": 2e-5})
    if CHAMPION == "XGBoost (tuned)":
        return Tabular(XGB(tree_method="hist", random_state=0, **search.best_params_)).fit(X, y_)
    if CHAMPION == "HistGradientBoosting":
        return Tabular(Probe(max_iter=300, random_state=0)).fit(X, y_)
    if CHAMPION == "Linear baseline":
        return Tabular(Base()).fit(X, y_)
    return fit_tfm(ROSTER[0], X, y_)

for m in range(MONTHS):
    label, mo, nxt = f"M{m + 1}", months[m], (months[m + 1] if m + 1 < MONTHS else None)
    live = metric(mo[TARGET_COL], score_of(state["model"], mo[FEATURES]))
    drifted = [f for f in MON_FEATURES if psi(state["reference"][f], mo[f]) > PSI_ALERT]
    decay = baseline_perf - live
    reasons = ([f"{len(drifted)} drifted feature(s): {', '.join(drifted[:3])}"] if drifted else []) + ([f"{PRIMARY} down {decay:.3f}"] if decay > DECAY_ALERT else [])
    audit("retrain-agent", f"{label} observe", f"{PRIMARY} {live:.4f} (release {baseline_perf:.4f}); {len(drifted)} features above PSI {PSI_ALERT}",
          ("RETRAIN PROPOSED" if nxt is not None else "flagged; no later month to verify on") if reasons else "no action")
    row = {"month": label, "live": live, "drifted": len(drifted), "action": "none", "version": state["version"]}
    if reasons and nxt is not None and human_gate(f"{label}: retrain because {'; '.join(reasons)}?", True):
        recent = pd.concat(months[: m + 1], ignore_index=True)
        keep_old = max(N_CONTEXT - len(recent), N_CONTEXT // 2)
        fresh = pd.concat([ctx.sample(min(keep_old, len(ctx)), random_state=0), recent], ignore_index=True)
        t0 = time.time(); challenger = refit_like_champion(fresh[FEATURES], fresh[TARGET_COL]); secs = time.time() - t0
        c_old, c_new = metric(nxt[TARGET_COL], score_of(state["model"], nxt[FEATURES])), metric(nxt[TARGET_COL], score_of(challenger, nxt[FEATURES]))
        audit("retrain-agent", f"{label} verify", f"on unseen {f'M{m + 2}'}: champion {c_old:.4f} vs challenger {c_new:.4f} (retrained in {secs:.0f}s on {len(fresh):,} rows)")
        win = c_new - c_old >= PROMOTE_MARGIN
        promoted = win and human_gate(f"Promote the challenger (+{c_new - c_old:.4f} {PRIMARY})?", True)
        if promoted:
            state.update(model=challenger, version=state["version"] + 1, reference=fresh)
        audit("retrain-agent", f"{label} promote", f"margin required {PROMOTE_MARGIN}", f"PROMOTED v{state['version']}" if promoted else "KEPT champion")
        state["events"].append({"month": label, "reasons": reasons, "rows": len(fresh), "champion_next": c_old, "challenger_next": c_new, "promoted": promoted, "seconds": secs})
        row.update(action="promoted challenger" if promoted else "retrained, kept champion", version=state["version"])
    state["history"].append(row)

hist = pd.DataFrame(state["history"])
fig = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4], subplot_titles=(f"{PRIMARY} by month · the model in production", "Champion vs challenger on the next unseen month"))
fig.add_trace(go.Scatter(x=psi_tab.columns, y=perf_m, name="original model, never retrained", line=dict(color=GREY, dash="dot")), 1, 1)
fig.add_trace(go.Scatter(x=hist.month, y=hist.live, name="model in production (with the agent)", mode="lines+markers", line=dict(color=BLUE, width=3)), 1, 1)
fig.add_hline(y=baseline_perf, line_dash="dot", line_color=TEAL, row=1, col=1)
for e in state["events"]:
    fig.add_vline(x=e["month"], line_color=ORANGE, line_dash="dash", row=1, col=1)
    fig.add_trace(go.Bar(x=[f"{e['month']} champion", f"{e['month']} challenger"], y=[e["champion_next"], e["challenger_next"]], marker_color=[GREY, BLUE], showlegend=False,
                         text=[f"{e['champion_next']:.4f}", f"{e['challenger_next']:.4f}"], textposition="outside"), 1, 2)
if state["events"]:
    vals = [v for e in state["events"] for v in (e["champion_next"], e["challenger_next"])]
    fig.update_yaxes(range=[min(vals) - 0.05, max(vals) + 0.03], row=1, col=2)
fig.update_layout(template="plotly_white", height=460, title=f"Retraining agent · model now at v{state['version']}", legend=dict(orientation="h", y=-0.18))
fig.show()
display(hist.style.hide(axis="index").format(precision=4))

## 9.3 Regenerated model card
The card is rewritten from the updated fact sheet, with a change log. The same grounding check runs again.

In [ ]:
# 📝 Model card v2 after the production period
changes = []
if state["events"]:
    n_prom = sum(e["promoted"] for e in state["events"]); e = state["events"][0]
    record("retraining", retraining_runs=len(state["events"]), promotions=n_prom, first_trigger_month=e["month"], first_trigger="; ".join(e["reasons"]),
           retraining_rows=e["rows"], champion_score_next_month=e["champion_next"], challenger_score_next_month=e["challenger_next"],
           outcome=f"{len(state['events'])} retraining run(s), {n_prom} promoted", human_approvals="retraining and promotion gates")
    for e in state["events"]:
        changes += [f"{e['month']}: retraining triggered ({'; '.join(e['reasons'])}); challenger trained on {e['rows']} rows.",
                    f"{e['month']}: on the next unseen month the champion scored {e['champion_next']:.4f} and the challenger {e['challenger_next']:.4f}; "
                    + ("challenger promoted after human approval." if e["promoted"] else "champion kept, the challenger did not clear the promotion margin.")]
else:
    record("retraining", outcome="no retraining was triggered during the monitoring period")
    changes = ["Monitoring period completed with no retraining trigger."]
record("monitoring", months_monitored=MONTHS, drift_feature=drift_feature, max_feature_psi=float(psi_tab.values.max()), final_month_score=float(hist.live.iloc[-1]))
FACTS["use_case"]["version"] = f"v{state['version']}" if state["version"] > 1 else "v1 (reviewed)"
CARD_V2, TRACE_V2, UNVERIFIED_V2 = generate_model_card(FACTS, changes=changes, version_label="v2")
show_card(CARD_V2, TRACE_V2, UNVERIFIED_V2)
with open("model_card_v2.md", "w") as f:
    f.write(CARD_V2)
import difflib
d = list(difflib.unified_diff(CARD_V1.splitlines(), CARD_V2.splitlines(), lineterm="", n=0))
print(f"\nmodel card diff v1 → v2: {sum(l.startswith('+') and not l.startswith('+++') for l in d)} lines added, "
      f"{sum(l.startswith('-') and not l.startswith('---') for l in d)} removed")

# 10 · Conclusions

In [ ]:
# 🧾 Build the plain-English conclusions from the recorded facts
P, S, U = FACTS["performance"], FACTS["selection"], FACTS["uncertainty"]
headline = f"{P['roc_auc']:.3f} ROC AUC (F1 {P['f1']:.3f} at a threshold of {P['threshold']:.2f})" if TASK == "classification" else f"R² {P['r2']:.3f}, typical error {P['mape_pct']:.0f}%, {P['within_20pct_share']:.0f}% of recommendations within 20% of the actual amount"
CONCLUSIONS = [
    f"**Data:** {FACTS['preparation']['features_before']} candidate features were reduced to {FACTS['features']['final_feature_count']} by a data preparation agent, with a human approval at every step.",
    f"**Model:** {S['candidates_succeeded']} models were raced; **{S['selected_model']}** was selected on validation data ({S['selection_metric']} {S['selected_validation_score']:.4f}; best classic model {S['best_classic_validation_score']:.4f}).",
    f"**Performance on unseen data:** {headline}.",
    f"**Uncertainty:** a {U['target_coverage']:.0%} coverage promise measured {U['measured_coverage']:.1%} on the test set.",
    f"**Fairness:** measured across {FACTS['fairness']['groups']} {FACTS['fairness']['attribute']} groups; the gaps are in section 7.2.",
    f"**Documentation:** the model card was written by {TRACE_V2['writer']}; {TRACE_V2['grounded_pct']:.0f}% of its numbers trace back to recorded facts.",
    f"**Production:** over {MONTHS} simulated months the monitor tracked drift on '{drift_feature}'; outcome: {FACTS['retraining']['outcome']}.",
    "**Next step:** the same workflow runs on the Lexsi platform with managed compute, a model registry and case-level audit.",
]

In [ ]:
# ✅ Conclusions in plain English, and the full audit trail
display(Markdown("## What this exercise showed\n\n" + "\n".join(f"- {line}" for line in CONCLUSIONS)))
trail = pd.DataFrame(AUDIT)
if len(trail):
    fig = px.histogram(trail, x="step", color="actor", color_discrete_sequence=PALETTE, title="Audit trail · logged agent steps and human decisions")
    fig.update_layout(template="plotly_white", height=360, xaxis_title=""); fig.show()
    display(trail.style.hide(axis="index").set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))